<a href="https://colab.research.google.com/github/csu-techhub/QOS-Labs/blob/main/Lab4.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 4 — Preparing the H₂ Configuration Mix
**Quantum Optimization and Simulation — VQE Laboratory Series**

Turn the VQE control knob and observe bonding/antibonding populations, using the **reduced two-qubit** encoding that Module 3 built and Module 4 will use. An extension at the end builds the same trial state in the full **four-spin-orbital** encoding.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. In this notebook every bitstring we *discuss* is written in the logical order `q0 q1 ...` (left to right), matching the lecture slides. The helper `q0_first(...)` converts Qiskit's labels to this order.

## Learning objectives
- Use the reduced two-qubit H₂ encoding.
- Prepare a superposition of bonding and antibonding configurations with $R_{YX}(\theta)R_{XY}(-\theta)$.
- Interpret the variational parameter $\theta$ as a mixing control.
- Verify that the circuit stays in the physical $|01\rangle$, $|10\rangle$ subspace.
- *(Extension)* Build the same trial state in the four-spin-orbital encoding as a UCC double excitation, $e^{\theta(T_D - T_D^\dagger)}|1010\rangle$, and compare the cost.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "pylatexenc" "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from scipy.linalg import expm

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096
# Helper functions
def run_counts(qc, shots=SHOTS, noise_model=None):
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=SEED).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

def probs_q0_first(counts):
    "Convert Qiskit counts into probabilities keyed by q0-first bitstrings."
    total = sum(counts.values())
    return {q0_first(k): v / total for k, v in counts.items()}

def amplitudes_q0_first(qc, tol=1e-9):
    "Nonzero statevector amplitudes keyed by q0-first bitstrings (real part shown when imaginary part is ~0)."
    sv = Statevector.from_instruction(qc)
    out = {}
    for k, a in sv.to_dict().items():
        if abs(a) > tol:
            out[q0_first(k)] = np.round(a.real, 4) if abs(a.imag) < tol else np.round(a, 4)
    return out

## Reduced encoding
H₂ has a bonding orbital $\sigma_g$ and an antibonding orbital $\sigma_u^*$. For the ground state, the only configuration change that matters is **both electrons moving together** between the two orbitals. So two qubits are enough:

| Reduced state ($q_0q_1$) | Meaning |
|---|---|
| $\lvert 00\rangle$ | not physical here |
| $\lvert 01\rangle$ | electrons in the bonding orbital only (Hartree–Fock reference) |
| $\lvert 10\rangle$ | electrons in the antibonding orbital only |
| $\lvert 11\rangle$ | not physical here |

We want a circuit $T(\theta)$ that mixes $|01\rangle \leftrightarrows |10\rangle$ and leaves $|00\rangle, |11\rangle$ unchanged. The slides show that $Y\!\otimes\!X - X\!\otimes\!Y$ connects exactly these two states, and that its exponential splits into

$$
R_{YX}(\theta)\,R_{XY}(-\theta) = e^{-i\frac{\theta}{2}Y_0X_1}\; e^{-i\frac{-\theta}{2}X_0Y_1},
$$

which acts as $|01\rangle \mapsto \cos\theta\,|01\rangle + \sin\theta\,|10\rangle$. Each factor is a CNOT–$R_z$–CNOT sandwich with basis changes ($HZH=X$, $SHZHS^\dagger=Y$).

*(The extension shows where this comes from: in four spin orbitals the two states are $|1010\rangle$ and $|0101\rangle$.)*

## Part A — An excitation-preserving ansatz

In [ ]:
def h2_ansatz(theta):
    qc = QuantumCircuit(2)

    # Prepare |01> (q0-first): the Hartree-Fock reference in the reduced encoding
    qc.x(1)

    # ------------------------------------------------
    # RYX(theta) = exp[-i theta/2 (Y0 X1)]
    # ------------------------------------------------
    qc.sdg(0)
    qc.h(0)
    qc.h(1)

    # ZZ rotation
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)

    # Undo basis change
    qc.h(0)
    qc.s(0)
    qc.h(1)

    # ------------------------------------------------
    # RXY(-theta) = exp[+i theta/2 (X0 Y1)]
    # ------------------------------------------------

    qc.h(0)
    qc.sdg(1)
    qc.h(1)

    # ZZ rotation with angle -theta
    qc.cx(0, 1)
    qc.rz(-theta, 1)
    qc.cx(0, 1)

    # Undo basis change
    qc.h(0)
    qc.h(1)
    qc.s(1)
    return qc

theta = 0.35
qc2 = h2_ansatz(theta)

display(qc2.draw("mpl"))
print("amplitudes (q0-first):", amplitudes_q0_first(qc2))
print(f"cos(theta) = {np.cos(theta):.4f},  sin(theta) = {np.sin(theta):.4f}")

> **Bitstring order matters here.** The bonding state $|01\rangle$ (q0-first, as in the slides) is reported by Qiskit as the key `"10"`. Reading `counts.get("01")` directly would mislabel the antibonding population as bonding. Below we always convert with `probs_q0_first`.

### NOTE
From the last cell, with $\theta = 0.35$ (or $\pi/9$), the probability amplitude for $|00\rangle$ (bonding) and $|11\rangle$ (antibonding) is 0.34 and 0.94, respectively.

What's the amplitudes on $|01\rangle$ and $|10\rangle$ for $\theta = \pi/4$ and $\theta = \pi/2$? Any amplitude on $|00\rangle$ and $|11\rangle$? Run the next cell.

In [ ]:
for th in [np.pi/4, np.pi/2]:
    print(f"theta = {th:.4f}:", amplitudes_q0_first(h2_ansatz(th)))

## Part B — Sample the configuration probabilities

In [ ]:
angles = np.linspace(0, np.pi/2, 9)
p01, p10, p_unphys = [], [], []

for th in angles:
    qc = h2_ansatz(th)
    qc.measure_all()
    probs = probs_q0_first(run_counts(qc, shots=SHOTS))
    p01.append(probs.get("01", 0))
    p10.append(probs.get("10", 0))
    p_unphys.append(probs.get("00", 0) + probs.get("11", 0))

plt.plot(angles, p01, "o-", label="P(01): bonding")
plt.plot(angles, p10, "s-", label="P(10): antibonding")
plt.plot(angles, p_unphys, "x:", label="P(00) + P(11): unphysical")
plt.xlabel("theta"); plt.ylabel("Probability"); plt.title("Reduced 2-qubit ansatz")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Expected:** the two probabilities exchange smoothly as $\theta$ changes, while `00` and `11` remain essentially absent.

### YOUR TURN
Find a value of `theta` that produces approximately 80% bonding and 20% antibonding. First predict it from $P(01)=\cos^2\theta$, then check by sampling.

In [ ]:
theta_guess = 0.0  # TODO
qc = h2_ansatz(theta_guess)
qc.measure_all()
probs = probs_q0_first(run_counts(qc, shots=8192))
print({k: round(v, 3) for k, v in sorted(probs.items())})

## Reflection
Is $\theta$ itself a probability?

No. $\theta$ is a circuit parameter/rotation angle. It controls amplitudes, and probabilities are obtained by squaring amplitude magnitudes. Depending on the gate convention, the probabilities follow trigonometric functions of $\theta$. Here $P_{\text{bonding}} = \cos^2\theta$.

---
# Extension — The full four-spin-orbital encoding
Where do $|01\rangle$ and $|10\rangle$ come from? This extension builds the same trial state on **four** qubits, one per spin orbital, as the UCC double excitation from the lecture, and shows why two qubits are enough for H₂.

## Part C1 — The four spin orbitals ("lanes")

| Qubit | Spin orbital | Lane |
|---|---|---|
| $q_0$ | bonding, spin up, $\sigma_g(\alpha)$ | 0 |
| $q_1$ | antibonding, spin up, $\sigma_u^*(\alpha)$ | 1 |
| $q_2$ | bonding, spin down, $\sigma_g(\beta)$ | 2 |
| $q_3$ | antibonding, spin down, $\sigma_u^*(\beta)$ | 3 |

We use `1 = occupied`, `0 = empty`. Electrons are the cars and spin orbitals are the lanes. All spin-up orbitals come first, then all spin-down orbitals, which is the same ordering Qiskit Nature uses.

> **Note:** the separate four-qubit Lab 4 notebook ordered the lanes as (bonding↑, bonding↓, antibonding↑, antibonding↓). This notebook follows the lecture slides, so the Hartree–Fock state is $|1010\rangle$ here.

### Hartree–Fock reference and the configurations of H₂
The Hartree–Fock (HF) reference $|\Phi_A\rangle = \sigma_g^2$ puts both electrons in the bonding orbital: one spin-up car in lane 0 and one spin-down car in lane 2.

In [ ]:
hf = QuantumCircuit(4)
hf.x(0)   # spin-up electron in sigma_g(alpha)
hf.x(2)   # spin-down electron in sigma_g(beta)

display(hf.draw("mpl"))
print("HF state (q0-first):", amplitudes_q0_first(hf))
print("Qiskit's own label :", Statevector.from_instruction(hf).to_dict())

Two electrons in four lanes allow $\binom{4}{2}=6$ occupation patterns. The ground state of H₂ has total spin zero (one spin-up and one spin-down electron), and no two electrons can share a lane (Pauli exclusion). That leaves **four** configurations: $|\Phi_A\rangle,\ |\Phi_B\rangle,\ |\Phi_C\rangle,\ |\Phi_D\rangle$.

In [ ]:
configurations = ["".join(map(str, bits)) for bits in product([0, 1], repeat=4) if sum(bits) == 2]
print("All two-electron configurations (q0 q1 q2 q3):", configurations)

# one electron among the spin-up lanes (q0, q1) and one among the spin-down lanes (q2, q3)
spin_zero = [c for c in configurations if c[0:2].count("1") == 1 and c[2:4].count("1") == 1]
print("Spin-zero configurations:", spin_zero)

## Part C2 — The 4-qubit UCC double-excitation ansatz
For H₂ the UCCSD operator keeps only the double excitation: the two single excitations $T_B, T_C$ give zero contribution because of the molecule's spatial (g/u) symmetry (see Optional C5). So the trial state is

$$
|\Psi(\theta)\rangle = e^{\theta\,(T_D - T_D^\dagger)}\,|1010\rangle,
\qquad T_D - T_D^\dagger = a_1^\dagger a_3^\dagger a_2 a_0 - a_0^\dagger a_2^\dagger a_3 a_1 .
$$

Fermionic operators cannot run on qubits directly. The **Jordan–Wigner** mapping turns $T_D - T_D^\dagger$ into a sum of **"eight"** Pauli strings.

In [ ]:
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

# A = T_D - T_D^dagger  (anti-Hermitian), lanes as in the table above
double_generator = FermionicOp(
    {
        "+_1 +_3 -_2 -_0":  1.0,   # a1† a3† a2 a0 : |1010> -> |0101>
        "+_0 +_2 -_3 -_1": -1.0,   # its Hermitian conjugate, with a minus sign
    },
    num_spin_orbitals=4,
)
jw_double = JordanWignerMapper().map(double_generator)

# U(theta) = exp(theta*A) = exp(-i theta G) with Hermitian G = iA
G_double = (1j * jw_double).simplify()
print("Hermitian generator G = i(T_D - T_D†)   (Qiskit labels: q3 q2 q1 q0)")
for label, c in G_double.to_list():
    print(f"  {c.real:+.3f}  {label}")

### From Pauli strings to a circuit
All **"eight"** strings **commute** with each other, so the exponential factorizes exactly into eight separate rotations:

$$
e^{-i\theta G} = \prod_{k=1}^{8} e^{-i\,\theta c_k P_k}, \qquad c_k = \pm\tfrac18 .
$$

Each rotation $e^{-i\frac{\phi}{2}P}$ is built exactly like the 2-qubit $R_{YX}$ and $R_{XY}$ blocks in the slides, only with a longer CNOT chain:
1. **Basis change:** on each qubit with $X$ apply $H$, and with $Y$ apply $S^\dagger$ then $H$ (because $HZH=X$ and $SHZHS^\dagger=Y$).
2. **CNOT ladder** $q_0\to q_1\to q_2\to q_3$, which collects the parity of all four qubits on $q_3$.
3. $R_z(\phi)$ on $q_3$: the "control knob".
4. Undo the ladder and the basis change.

In [ ]:
def append_pauli_rotation(qc, label, phi):
    "Append exp(-i phi/2 * P) for a Pauli label in Qiskit order (rightmost character = q0)."
    paulis = label[::-1]                       # now paulis[q] is the Pauli on qubit q
    active = [q for q, p in enumerate(paulis) if p != "I"]
    # 1. basis change into Z
    for q in active:
        if paulis[q] == "X":
            qc.h(q)
        elif paulis[q] == "Y":
            qc.sdg(q); qc.h(q)
    # 2. CNOT ladder, 3. Rz knob, 4. undo ladder
    for a, b in zip(active[:-1], active[1:]):
        qc.cx(a, b)
    qc.rz(phi, active[-1])
    for a, b in reversed(list(zip(active[:-1], active[1:]))):
        qc.cx(a, b)
    # undo basis change
    for q in active:
        if paulis[q] == "X":
            qc.h(q)
        elif paulis[q] == "Y":
            qc.h(q); qc.s(q)

def h2_ansatz_4q(theta, barriers=False):
    qc = QuantumCircuit(4)
    qc.x(0); qc.x(2)                           # Hartree-Fock |1010>
    if barriers: qc.barrier()
    for label, c in G_double.to_list():        # exp(-i theta c P) = exp(-i (2 theta c)/2 P)
        append_pauli_rotation(qc, label, 2 * theta * c.real)
        if barriers: qc.barrier()              # for drawing only: separates the 8 rotations
    return qc

theta = 0.35
qc4 = h2_ansatz_4q(theta)
# Drawn with barriers so each of the 8 Pauli rotations is visible as one block
h2_ansatz_4q(theta, barriers=True).draw("mpl", fold=45)

### YOUR TURN
Each rotation above uses 6 CNOTs. Therefore, a total of 48 CNOTs. Quantum circuit optimizer, called transpiler, (optimization level 3) can bring it down because consecutive rotations share identical CNOT ladders: the "undo" ladder of one rotation and the ladder of the next cancel wherever the basis changes between them also cancel. Run the cell below to see how many?

In [ ]:
# YOUR TURN: compare CNOT counts before and after optimization.
opt = transpile(qc4, basis_gates=["cx", "rz", "sx", "x"], optimization_level=3)
print("written:", qc4.count_ops().get("cx", 0), " optimized:", opt.count_ops().get("cx", 0))

## Part C3 — Sample the 4-qubit configuration probabilities

In [ ]:
angles = np.linspace(0, np.pi/2, 9)
p_A, p_D, p_other = [], [], []

for th in angles:
    qc = h2_ansatz_4q(th)
    qc.measure_all()
    probs = probs_q0_first(run_counts(qc, shots=SHOTS))
    p_A.append(probs.get("1010", 0))     # both electrons bonding (HF)
    p_D.append(probs.get("0101", 0))     # both electrons antibonding
    p_other.append(1 - probs.get("1010", 0) - probs.get("0101", 0))

plt.plot(angles, p_A, "o-", label="P(1010): bonding, $\\Phi_A$")
plt.plot(angles, p_D, "s-", label="P(0101): antibonding, $\\Phi_D$")
plt.plot(angles, p_other, "x:", label="all other bitstrings")
plt.xlabel("theta"); plt.ylabel("Probability"); plt.title("4-qubit UCC double-excitation ansatz")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Expected:** $P(1010)=\cos^2\theta$ and $P(0101)=\sin^2\theta$ exchange smoothly, and the other 14 bitstrings never appear. The circuit conserves electron number *and* spin.

### NOTE
What do we *give up* by using the reduced encoding? (Think about which configurations the 2-qubit circuit can no longer represent.)

**Hint:** The reduced encoding cannot represent the singly excited configurations $\Phi_B, \Phi_C$ (or any state where the spin-up and spin-down electrons are in different orbitals). That is fine for ground-state H₂, but it would be wrong for excited states, broken spin symmetry, or larger molecules.